# 04 — Early Exercise Premium (H5)

**H5.** The market price of American-style options exceeds the
Black-Scholes European-equivalent price by an amount the CRR binomial
model predicts reasonably well, with the premium concentrated in ITM puts
on dividend-paying names.
- H0: American premium is negligible / binomial doesn't track it
- H1: premium is material and binomial tracks it better than BS

Real SPY and AAPL listed options are American-style, so the market mid
price already embeds an early-exercise premium that the *European*
Black-Scholes formula structurally cannot price. We compare, contract by
contract: `market mid vs. BS European` and `market mid vs. binomial
American`, using each contract's own market-implied vol (so the comparison
isolates the exercise-style effect rather than a vol-fit effect).

### A design trap to avoid: circularity in the vol input

Each contract's own Brent-solved implied vol is, by construction, exactly
the sigma that makes `BS(sigma) = market mid`. If we priced both BS and
the binomial model using *each contract's own* implied vol, `market minus
BS gap` would be tautologically ~0 for every single contract -- not
because BS is a good European approximation for an American option, but
because we solved for the vol that forces agreement. That comparison
would prove nothing.

Instead we fit one **smile curve per expiry using only OTM contracts**
(OTM calls for K>S, OTM puts for K<S -- the standard market convention,
since OTM quotes have negligible early-exercise premium and dominate
liquidity). That fitted curve is then applied to *every* contract at that
expiry, including ITM ones, to get a single non-circular vol input. Note
this needs no extrapolation: OTM calls span the K>S side and OTM puts span
the K<S side, so together they already cover the full moneyness range --
ITM points (which sit on the *opposite* side from their matching OTM set)
fall inside the fitted domain, not outside it.

In [ ]:
import sys, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats

from data_loader import latest_pull
from implied_vol import add_implied_vol
from diagnostics import early_exercise_premium

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)


def is_otm(row):
    return (row["option_type"] == "call" and row["log_moneyness"] >= 0) or \
           (row["option_type"] == "put" and row["log_moneyness"] <= 0)


def fit_smile(grp):
    # fit the OTM-only quadratic, return (model, prediction-for-whole-group)
    otm = grp[grp["is_otm"]]
    X_fit = sm.add_constant(pd.DataFrame({"lm": otm.log_moneyness, "lm2": otm.log_moneyness ** 2}))
    model = sm.OLS(otm.iv_computed, X_fit).fit()
    X_all = sm.add_constant(pd.DataFrame({"lm": grp.log_moneyness, "lm2": grp.log_moneyness ** 2}))
    pred = model.predict(X_all)
    pred.index = grp.index
    return model, pred


results = {}
smile_fit_diagnostics = []
for tkr in ["SPY", "AAPL"]:
    summary, df = latest_pull(tkr)
    out, iv_report = add_implied_vol(df)
    valid = out[out.iv_success].copy()
    valid["is_otm"] = valid.apply(is_otm, axis=1)

    sigma_smile = pd.Series(index=valid.index, dtype=float)
    for expiry, grp in valid.groupby("expiry"):
        n_otm = int(grp["is_otm"].sum())
        model, pred = fit_smile(grp)
        sigma_smile.loc[pred.index] = pred.values
        smile_fit_diagnostics.append({
            "ticker": tkr, "expiry": expiry, "n_otm_used_for_fit": n_otm,
            "n_total_priced": len(grp), "R2": model.rsquared,
            "resid_std": float(model.resid.std()),
        })
    valid["sigma_smile"] = sigma_smile

    # feed the externally-fit smile vol into the pricers, not each
    # contract's own solved IV, to avoid the circularity above
    priced = valid.copy()
    priced["iv_computed"] = priced["sigma_smile"]

    ee = early_exercise_premium(priced, N=500)
    ee["ticker"] = tkr
    ee = ee.merge(valid[["ticker", "expiry", "strike", "option_type", "log_moneyness", "T_years", "is_otm"]],
                   on=["ticker", "expiry", "strike", "option_type"], how="left")
    results[tkr] = {"summary": summary, "ee": ee}
    print(f"{tkr}: {len(ee)} contracts, q_used={summary['dividend_yield']['q_used']:.4f}")

### Smile-fit quality (feeds directly into the premium numbers below)

Unlike the H2 regression, this OTM-only fit was previously used without
reporting whether it's actually a good fit -- if it isn't, the early-
exercise premium numbers built on top of it aren't trustworthy either. R2
and residual std below make that checkable rather than assumed.

In [ ]:
smile_diag_df = pd.DataFrame(smile_fit_diagnostics)
smile_diag_df

## Moneyness buckets (ITM / ATM / OTM, exercise-style aware)

In [ ]:
def moneyness_bucket(row):
    lm = row["log_moneyness"]
    if abs(lm) < 0.03:
        return "ATM"
    if row["option_type"] == "call":
        return "ITM" if lm < 0 else "OTM"
    else:  # put
        return "ITM" if lm > 0 else "OTM"

for tkr, d in results.items():
    d["ee"]["moneyness_bucket"] = d["ee"].apply(moneyness_bucket, axis=1)

all_ee = pd.concat([d["ee"].assign(ticker=tkr) for tkr, d in results.items()], ignore_index=True)
all_ee = all_ee.dropna(subset=["binomial_early_exercise_premium"])
print(f"{len(all_ee)} contracts with a valid premium computation")

## Cross-model comparison: market gap to BS vs. market gap to binomial

In [ ]:
summary_table = all_ee.groupby(["ticker", "option_type"]).agg(
    n=("strike", "size"),
    mean_market_minus_bs_gap=("market_minus_bs_gap", "mean"),
    mean_market_minus_binomial_gap=("market_minus_binomial_gap", "mean"),
    mean_binomial_ee_premium=("binomial_early_exercise_premium", "mean"),
).reset_index()
summary_table["gap_reduction_pct"] = 100 * (
    summary_table.mean_market_minus_bs_gap.abs() - summary_table.mean_market_minus_binomial_gap.abs()
) / summary_table.mean_market_minus_bs_gap.abs()
summary_table

If the binomial American price tracks the market better than the BS
European price, `mean_market_minus_binomial_gap` should be closer to zero
(in absolute value) than `mean_market_minus_bs_gap`, i.e.
`gap_reduction_pct` should be positive.

**Caveat on this specific table, stated up front rather than after seeing
whether it looks good:** these are *raw-dollar* gaps averaged across an
entire option type/ticker, pooling contracts whose prices span cents
(deep OTM) to hundreds of dollars (deep ITM). A handful of deep-ITM, high
-intrinsic-value contracts can dominate a raw-dollar mean and swamp the
signal from the many more numerous, cheaper, near-the-money contracts.
The moneyness-bucketed table below, expressed as **premium as a percent
of price**, is the more reliable read on whether binomial tracks the
market better than BS -- check both before drawing a conclusion, and if
they disagree, say so rather than quoting whichever one looks better.

## Premium concentration by moneyness bucket

In [ ]:
bucket_table = all_ee.groupby(["ticker", "option_type", "moneyness_bucket"]).agg(
    n=("strike", "size"),
    mean_ee_premium=("binomial_early_exercise_premium", "mean"),
    mean_ee_premium_pct_of_price=("binomial_early_exercise_premium",
                                    lambda x: 100 * x.mean() / all_ee.loc[x.index, "market_mid"].mean()),
).reset_index().sort_values(["ticker", "option_type", "moneyness_bucket"])
bucket_table

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, tkr in zip(axes, ["SPY", "AAPL"]):
    sub = all_ee[all_ee.ticker == tkr]
    for opt, marker in [("call", "o"), ("put", "^")]:
        s = sub[sub.option_type == opt]
        ax.scatter(s.log_moneyness, s.binomial_early_exercise_premium, s=14, alpha=0.5, marker=marker, label=opt)
    ax.axhline(0, color="gray", lw=0.8)
    ax.set_xlabel("log-moneyness ln(K/S)")
    ax.set_ylabel("binomial early-exercise premium")
    ax.set_title(f"{tkr} (q_used={results[tkr]['summary']['dividend_yield']['q_used']:.4f})")
    ax.legend()
plt.tight_layout()
plt.savefig("../results/figures/h5_ee_premium_by_moneyness.png", dpi=120)
plt.show()

## Statistical test: is the ITM-put premium significantly larger than other buckets?

In [ ]:
itm_put_premium = all_ee[(all_ee.option_type == "put") & (all_ee.moneyness_bucket == "ITM")]["binomial_early_exercise_premium"]
other_premium = all_ee[~((all_ee.option_type == "put") & (all_ee.moneyness_bucket == "ITM"))]["binomial_early_exercise_premium"]

t_stat, p_value = stats.ttest_ind(itm_put_premium, other_premium, equal_var=False)
print(f"ITM put mean premium: {itm_put_premium.mean():.4f}  (n={len(itm_put_premium)})")
print(f"All other contracts mean premium: {other_premium.mean():.4f}  (n={len(other_premium)})")
print(f"Contract-level Welch t-test: t={t_stat:.3f}, p={p_value:.3e}")

**Independence caveat.** The test above treats each contract as an
independent observation, but contracts sharing a (ticker, expiry) share
the same underlying spot, the same smile fit, and overlapping
microstructure noise -- they are not independent draws, so that p-value
is optimistic. As a more conservative supplement, aggregate to one mean
premium per (ticker, expiry, bucket) group first, so each group -- a
genuinely distinct sample of market conditions -- counts once.

In [ ]:
group_means = all_ee.groupby(["ticker", "expiry", "option_type", "moneyness_bucket"]
                              )["binomial_early_exercise_premium"].mean().reset_index()

itm_put_group_means = group_means[(group_means.option_type == "put") &
                                    (group_means.moneyness_bucket == "ITM")]["binomial_early_exercise_premium"]
other_group_means = group_means[~((group_means.option_type == "put") &
                                    (group_means.moneyness_bucket == "ITM"))]["binomial_early_exercise_premium"]

t_stat_grp, p_value_grp = stats.ttest_ind(itm_put_group_means, other_group_means, equal_var=False)
print(f"Group-level (one point per ticker/expiry/bucket) ITM put mean: {itm_put_group_means.mean():.4f} "
      f"(n={len(itm_put_group_means)})")
print(f"Group-level other mean: {other_group_means.mean():.4f}  (n={len(other_group_means)})")
print(f"Group-level Welch t-test: t={t_stat_grp:.3f}, p={p_value_grp:.3e}")
print()
print("This is a much smaller, much more honest sample size (n=6 ticker/expiry combos "
      "per side at most) -- read this p-value, not the contract-level one, as the "
      "primary claim of statistical significance, and treat the contract-level test "
      "as a secondary, higher-powered-but-optimistic check.")

## Persist results for the report

In [ ]:
summary_table.to_csv("../results/tables/h5_cross_model_gap_summary.csv", index=False)
bucket_table.to_csv("../results/tables/h5_premium_by_moneyness_bucket.csv", index=False)
smile_diag_df.to_csv("../results/tables/h5_smile_fit_diagnostics.csv", index=False)
pd.DataFrame([{"t_stat": t_stat, "p_value": p_value,
               "itm_put_mean_premium": itm_put_premium.mean(), "itm_put_n": len(itm_put_premium),
               "other_mean_premium": other_premium.mean(), "other_n": len(other_premium),
               "t_stat_group_level": t_stat_grp, "p_value_group_level": p_value_grp,
               "itm_put_group_mean": itm_put_group_means.mean(), "itm_put_group_n": len(itm_put_group_means),
               "other_group_mean": other_group_means.mean(), "other_group_n": len(other_group_means)}]
             ).to_csv("../results/tables/h5_ttest.csv", index=False)
print("saved h5 tables to results/tables/")

## H5 verdict

Read directly off the tables and test above:

- **Materiality (H0 vs H1, first half):** whether `mean_binomial_ee_premium`
  is economically meaningful (not just statistically nonzero) for ITM puts,
  from the bucket table.
- **Binomial tracking:** whether `gap_reduction_pct` in the cross-model
  table is positive and large -- i.e. whether switching from BS European to
  binomial American materially closes the gap to the observed market mid.
- **Concentration in ITM puts on dividend payers:** compare the ITM-put row
  across SPY (higher `q_used`) vs. AAPL (lower `q_used`) in the bucket
  table, and the Welch t-test above for whether ITM puts are statistically
  distinguishable from the rest of the surface.

Do not round up a partial result: if the premium is material but the
binomial model doesn't clearly track the market gap better than BS (e.g.
because both are using an approximate continuous dividend yield or a
single-day snapshot vol), or if AAPL's much smaller dividend yield still
shows an ITM-put premium of similar size to SPY's, say so plainly in
`report/report.md` rather than asserting the hypothesis is confirmed.